# 01 – Deutschland: Temperatur und Niederschlag

Dieses Notebook rechnet das Gebietsmittel für Deutschland **selbst** aus Stationsdaten nach und vergleicht es mit dem offiziellen Wert des Deutschen Wetterdienstes (DWD).

Methode (siehe `SPEC.md`, Abschnitt 3):
1. **Anomalien** je Station und Kalendermonat gegenüber der Referenzperiode (Standard 1951–1980),
2. **Gitterung** auf 1°×1°-Zellen, damit dichte Stationsnetze nicht dominieren,
3. **Flächengewichtung** mit dem Kosinus der Breite,
4. **Jahresmittel** nur aus vollständigen Jahren.

Voraussetzung: `uv run klima laden` (Rohdaten). Die Aufbereitung zu Parquet erfolgt bei Bedarf automatisch.

In [ ]:
%matplotlib inline
import pandas as pd

from klima import auswertungen, einlesen
from klima import deutschland as de
from klima.anomalien import standard_referenzperiode
from klima.grafik import interaktiv, statisch
from klima.trend import linearer_trend

pd.set_option("display.precision", 2)
REFERENZ = standard_referenzperiode()
REFERENZ

## Datengrundlage

In [ ]:
dwd_stationen = einlesen.dwd_stationen("monat")
ghcnm_stationen = einlesen.ghcnm_stationen("qcf", laender=["GM"])
print(f"DWD-Stationen mit Monatswerten: {len(dwd_stationen)}")
print(f"GHCNm-Stationen in Deutschland: {len(ghcnm_stationen)}")
dwd_stationen.head()

## Temperatur: vier Wege zum Gebietsmittel

| Reihe | Quelle | Methode |
|---|---|---|
| DWD-Gebietsmittel (offiziell) | DWD | Interpolation auf 1-km-Raster |
| DWD-Stationen | DWD-Monatswerte | eigene Gitterung |
| GHCNm homogenisiert | NOAA (QCF) | eigene Gitterung |
| GHCNm unbereinigt | NOAA (QCU) | eigene Gitterung |

In [ ]:
temperatur = de.temperatur_jahresanomalien(REFERENZ).loc[1881:]
temperatur.tail()

In [ ]:
trends = pd.DataFrame(
    {name: vars(linearer_trend(temperatur[name], von=1951)) for name in temperatur}
).T[["steigung_pro_dekade", "unsicherheit_95", "von", "bis"]]
trends

In [ ]:
statisch.zeitreihen(
    temperatur,
    "Temperatur in Deutschland",
    referenz=REFERENZ,
    farben=auswertungen.FARBEN_QUELLEN,
    untertitel=f"Jahresmittel, Abweichung von {REFERENZ[0]}–{REFERENZ[1]}",
);

Interaktiv – Reihen lassen sich über die Legende ein- und ausblenden, beim Überfahren erscheinen die Werte:

In [ ]:
interaktiv.zeitreihen(
    temperatur,
    "Temperatur in Deutschland",
    referenz=REFERENZ,
    farben=auswertungen.FARBEN_QUELLEN,
)

### Wie gut trifft die eigene Berechnung den DWD?

Die Differenz zum offiziellen Gebietsmittel zeigt zweierlei:
- Aus **DWD-Stationen** gerechnet liegt die Abweichung meist unter 0,05 °C – die einfache Gittermethode reicht für Deutschland aus.
- Bei **GHCNm homogenisiert** ist die Vergangenheit (vor ca. 1950) kälter als beim DWD: Die Homogenisierung der NOAA korrigiert frühe Werte nach unten und erhöht damit den langfristigen Erwärmungstrend. Die unbereinigten Daten liegen näher am DWD. Diesen Effekt untersuchen wir global in M4.

In [ ]:
abweichung = temperatur.drop(columns=de.DWD_OFFIZIELL).sub(temperatur[de.DWD_OFFIZIELL], axis=0)
abweichung.loc[1951:].agg(["mean", "std"])

In [ ]:
statisch.zeitreihen(
    abweichung,
    "Eigene Berechnung minus DWD",
    glaettung=None,
    farben=auswertungen.FARBEN_QUELLEN,
    untertitel="Jahresmittel, °C",
);

## Warming Stripes

In [ ]:
statisch.warming_stripes(temperatur[de.DWD_OFFIZIELL], "Deutschland 1881–heute");

## Niederschlag

Niederschlag wird **relativ** ausgewertet (Prozent des Mittels der Referenzperiode), weil sich die Mengen regional stark unterscheiden.

In [ ]:
niederschlag = de.niederschlag_jahresanomalien(REFERENZ).loc[1881:]
print(linearer_trend(niederschlag[de.DWD_OFFIZIELL], von=1951).text("%"))
niederschlag.corr()

In [ ]:
statisch.balken_relativ(
    niederschlag[de.DWD_OFFIZIELL],
    "Niederschlag in Deutschland",
    untertitel=f"Jahressumme, Abweichung von {REFERENZ[0]}–{REFERENZ[1]} in Prozent",
);

## Trends je Station

In [ ]:
trends_temperatur = de.stationstrends(von=1951, wert="tmittel")
median = trends_temperatur.trend.median()
print(f"{len(trends_temperatur)} Stationen, Median {median:.2f} °C/Dekade")
interaktiv.stationskarte(trends_temperatur, "trend", "Temperaturtrend seit 1951", "°C/Dekade")

In [ ]:
trends_niederschlag = de.stationstrends(von=1951, wert="niederschlag")
trends_niederschlag.sort_values("trend").head(5)[["stations_id", "name", "hoehe", "trend", "jahre"]]

Auffällig: Die stärksten Abnahmen zeigen Bergstationen wie der **Wendelstein** (1.832 m). Hier sind Messprobleme (Windverluste, besonders bei Schnee; Gerätewechsel) wahrscheinlicher als ein echtes Klimasignal – ein Beispiel für die in der SPEC genannten Fallstricke.

## Einzelstation: Potsdam

In [ ]:
potsdam = auswertungen.station("03987")
{name: t.text("%" if name == "Niederschlag" else "°C") for name, t in potsdam.trends.items()}

In [ ]:
statisch.warming_stripes(potsdam.tabellen["temperatur"]["anomalie"], "Potsdam");